# [실습] LangChain의 다양한 채팅 템플릿



실제 어플리케이션 환경에서는 프롬프트를 매번 다시 쓰는 대신,  
프롬프트의 템플릿을 구성하고, 입력 변수의 공간을 할당하여 일관성 있는 입력을 전달합니다.  


LangChain은 다양한 형태의 프롬프트 템플릿 기능을 지원합니다.

## 실습 환경 설정

기본 라이브러리를 설치합니다.

In [1]:
!pip install rich pandas langchain_community openai langchain langchain_openai langchain_google_genai -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 96.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 120.5/120.5 kB 15.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 70.7/70.7 kB 9.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 76.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 558.3/558.3 kB 49.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.7/61.7 kB 7.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 9.2 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.


환경 변수 파일을 불러옵니다.

In [2]:
import os
from dotenv import load_dotenv
# OPENAI_API_KEY, GOOGLE_API_KEY
load_dotenv(override=True)

True

LLM을 정의합니다.   
API 키가 필요하므로, 실습 환경에 맞게 선택해 주세요.

In [3]:
from langchain_openai import ChatOpenAI
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.rate_limiters import InMemoryRateLimiter

# Gemini: 무료 API 사용량 존재
# 안정적 서빙을 위해 분당 10개 설정
# 즉, 초당 약 0.167개 요청 (10/60)
# `https://aistudio.google.com/`에서 모델별 사용량 확인

rate_limiter = InMemoryRateLimiter(
    requests_per_second=0.167,  # 분당 10개 요청
    check_every_n_seconds=0.1,  # 100ms마다 체크
    max_bucket_size=10,  # 최대 버스트 크기
)


# LLM 초기화
llm = ChatOpenAI(model='gpt-5.5', temperature=0.3, max_tokens=8192)
llm_gemini = ChatGoogleGenerativeAI(model='gemini-2.5-flash', temperature=0.7, max_tokens=8192,
                                rate_limiter=rate_limiter)

response = llm_gemini.invoke("안녕?")
response

AIMessage(content='안녕하세요! 반갑습니다. 😊', additional_kwargs={}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--019f4646-74b4-7361-b256-60fe3ba0ebd9-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 4, 'output_tokens': 27, 'total_tokens': 31, 'input_token_details': {'cache_read': 0}, 'output_token_details': {'reasoning': 22}})

In [7]:
from rich import print as rprint
rprint(response)

AIMessageChunk(
    content='',
    additional_kwargs={},
    response_metadata={},
    id='lc_run--019f4646-90b5-73c1-9d22-565502ca9f59',
    tool_calls=[],
    invalid_tool_calls=[],
    tool_call_chunks=[],
    chunk_position='last'
)

## LangChain의 Prompt

랭체인에서 LLM에 프롬프트를 입력하는 방법은 아래 3가지가 있습니다.

1. 문자열 그대로 입력하기 (이전 방법과 동일하게)
2. 메시지 클래스 리스트 입력하기
3. 프롬프트 템플릿 구성하기

### 1) 문자열 그대로 입력하기   

문자열을 `invoke()`를 통해 전달합니다.   

In [4]:
prompt = '''
역사상 가장 매력적이었던 악역이 나오는 영화 5개를 알려주세요.
주연 배우와 명대사, 매력적인 이유도 한 줄로 설명하세요.
출력의 마지막에는 전체 내용을 표로 표현하세요.'''

result = llm.stream(prompt)
for response in result:
    print(response.content, end='')

아래 5편은 영화사에서 특히 “악역의 존재감”이 강렬하다고 자주 언급되는 작품들입니다.

1. **《다크 나이트》 — 조커 / 히스 레저**  
   - **명대사:** “Why so serious?”  
   - **매력적인 이유:** 예측 불가능한 광기와 철학적 혼돈으로 영웅보다 더 강렬한 존재감을 남겼습니다.

2. **《양들의 침묵》 — 한니발 렉터 / 앤서니 홉킨스**  
   - **명대사:** “I ate his liver with some fava beans and a nice Chianti.”  
   - **매력적인 이유:** 지적이고 우아하지만 섬뜩한 이중성으로 관객을 압도합니다.

3. **《스타워즈: 제국의 역습》 — 다스 베이더 / 데이비드 프로즈·제임스 얼 존스**  
   - **명대사:** “No, I am your father.”  
   - **매력적인 이유:** 압도적인 비주얼, 목소리, 비극적 서사가 결합된 상징적인 악역입니다.

4. **《노인을 위한 나라는 없다》 — 안톤 쉬거 / 하비에르 바르뎀**  
   - **명대사:** “Call it.”  
   - **매력적인 이유:** 감정 없는 살인 기계처럼 보이지만 자신만의 규칙을 가진 공포 그 자체입니다.

5. **《바스터즈: 거친 녀석들》 — 한스 란다 / 크리스토프 왈츠**  
   - **명대사:** “That’s a bingo!”  
   - **매력적인 이유:** 유머, 지성, 잔혹함을 동시에 지닌 대화형 악역의 정점입니다.

| 영화 | 악역 | 배우 | 명대사 | 매력적인 이유 |
|---|---|---|---|---|
| 《다크 나이트》 | 조커 | 히스 레저 | “Why so serious?” | 예측 불가능한 광기와 철학적 혼돈으로 영웅보다 더 강렬한 존재감을 남겼습니다. |
| 《양들의 침묵》 | 한니발 렉터 | 앤서니 홉킨스 | “I ate his liver with some fava beans and a nice Chianti.” | 지적이고

In [ ]:
result = llm_gemini.stream(prompt)
for response in result:
    print(response.content, end='')

역사상 가장 매력적이었던 악역이 나오는 영화 5편을 선정하고, 그들의 매력적인 이유를 설명해 드립니다.

---

### 역사상 가장 매력적인 악역 영화 5선

1.  **영화 제목:** 다크 나이트 (The Dark Knight, 2008)
    *   **악역 이름:** 조커 (Joker)
    *   **주연 배우:** 히스 레저 (Heath Ledger)
    *   **명대사:** "Why so serious?" (왜 그렇게 진지해?)
    *   **매력적인 이유:** 예측 불가능한 광기와 혼돈을 통해 기존 질서를 파괴하며, 깊은 철학적 질문을 던지는 압도적인 존재감으로 관객을 사로잡습니다.

2.  **영화 제목:** 양들의 침묵 (The Silence of the Lambs, 1991)
    *   **악역 이름:** 한니발 렉터 (Hannibal Lecter)
    *   **주연 배우:** 안소니 홉킨스 (Anthony Hopkins)
    *   **명대사:** "A census taker once tried to test me. I ate his liver with some fava beans and a nice Chianti." (나는 한 번도 인구조사관에게 시험 당한 적이 없다. 그의 간을 파바 콩과 맛있는 끼안티 와인과 함께 먹었지.)
    *   **매력적인 이유:** 고도로 지적인 통찰력과 섬뜩한 교양을 겸비하며 인간 본성의 어두운 면을 탐구하는, 매혹적인 사이코패스의 전형입니다.

3.  **영화 제목:** 바스터즈: 거친 녀석들 (Inglourious Basterds, 2009)
    *   **악역 이름:** 한스 란다 (Hans Landa)
    *   **주연 배우:** 크리스토프 왈츠 (Christoph Waltz)
    *   **명대사:** "Ooh, that's a bingo! Is that how you say it? 'That's a bingo'?" (오, 빙고! 그렇게 말하는 건가? '빙고

### 2) Message 클래스 전달하기   
클래스를 직접 생성하고 전달합니다.  
`HumanMessage, SystemMessage, AIMessage` 클래스의 리스트를 전달하면 됩니다.

In [8]:
from langchain_core.messages import HumanMessage, SystemMessage, AIMessage

messages=[]

messages.append(SystemMessage('당신은 매우 창의력이 뛰어납니다.'))
messages.append(HumanMessage('엔비디아와 XAI가 합병한다면 어떤 이름이 좋을까요?'))

print('======= messages=====')
for message in messages :
  rprint(message)
print('=====================')
result = llm.invoke(messages)
rprint(result)

======= messages=====


SystemMessage(content='당신은 매우 창의력이 뛰어납니다.', additional_kwargs={}, response_metadata={})

HumanMessage(
    content='엔비디아와 XAI가 합병한다면 어떤 이름이 좋을까요?',
    additional_kwargs={},
    response_metadata={}
)

AIMessage(
    content='엔비디아(NVIDIA)와 xAI가 합병한다면, 개인적으로는 **“Nexvidia”**가 가장 그럴듯해 보입니다.\n\n### 추천
이름 후보\n\n1. **Nexvidia**\n   - **NVIDIA + xAI + Next** 느낌\n   - “차세대 AI 컴퓨팅 기업”이라는 인상이 강함\n  
- 발음도 비교적 자연스럽고 글로벌 브랜드로 쓰기 좋음\n\n2. **xVIDIA**\n   - xAI의 **x**와 NVIDIA를 결합\n   - 가장 
직관적이고 강렬함\n   - 다만 너무 단순해서 기존 NVIDIA의 서브브랜드처럼 보일 수 있음\n\n3. **NeuralX**\n   - 
신경망, AI, 미지의 가능성을 모두 담은 이름\n   - 합병 후 AI 중심 기업 이미지를 강조하기 좋음\n\n4. **OmniCore 
AI**\n   - GPU, 데이터센터, 로봇, 자율주행, 생성형 AI까지 아우르는 느낌\n   - “모든 지능의 핵심 인프라”라는 의미를 
줄 수 있음\n\n5. **NVX**\n   - NVIDIA의 **NV** + xAI의 **X**\n   - 짧고 강력한 티커/브랜드 느낌\n   - 실제 합병 
법인명이나 제품 플랫폼명으로 잘 어울림\n\n6. **GrokForce**\n   - xAI의 Grok과 NVIDIA의 연산 파워 이미지를 결합\n   
- 조금 더 제품명이나 AI 슈퍼컴퓨터 이름에 가까움\n\n7. **Xenvidia**\n   - xAI의 X와 NVIDIA를 자연스럽게 융합\n   - 
미래적이고 SF적인 느낌\n   - 단, 발음이 약간 낯설 수 있음\n\n8. **AetherX**\n   - “보이지 않는 지능 인프라” 같은 
고급스러운 이미지\n   - 직접적인 합성어는 아니지만 프리미엄 AI 기업명으로 좋음\n\n### 제일 좋은 
선택\n\n브랜드명으로는 **Nexvidia**  \n약칭이나 주식 티커 느낌으로는 **NVX**  \nAI 플랫폼명으로는 **NeuralX**가 
가장 좋아 보입니다.\n\n한 줄로 정리하면:\n\n> **합병 회사명: Nexvidia**  \n> **AI 플랫폼명: NeuralX**  \n> 
**칩/인프라 브랜드명: NVX**',
    additional_kwargs={'refusal': None},
    response_metadata={
        'token_usage': {
            'completion_tokens': 578,
            'prompt_tokens': 41,
            'total_tokens': 619,
            'completion_tokens_details': {
                'accepted_prediction_tokens': 0,
                'audio_tokens': 0,
                'reasoning_tokens': 46,
                'rejected_prediction_tokens': 0
            },
            'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}
        },
        'model_provider': 'openai',
        'model_name': 'gpt-5.5-2026-04-23',
        'system_fingerprint': None,
        'id': 'chatcmpl-DzfelM1KprPdxyJCLVYZZ80RLA7z8',
        'service_tier': 'default',
        'finish_reason': 'stop',
        'logprobs': None
    },
    id='lc_run--019f4647-5340-7dc1-887d-968aa200a00c-0',
    tool_calls=[],
    invalid_tool_calls=[],
    usage_metadata={
        'input_tokens': 41,
        'output_tokens': 578,
        'total_tokens': 619,
        'input_token_details': {'audio': 0, 'cache_read': 0},
        'output_token_details': {'audio': 0, 'reasoning': 46}
    }
)

AIMessage를 함께 전달하는 방식으로, 멀티-턴 대화를 수행할 수 있습니다.

In [9]:
messages.append(result) # AIMessage 추가
messages.append(HumanMessage('이 중에서 당신이 꼽은 가장 좋은 이름은 뭔가요?'))

result = llm.invoke(messages)
rprint(result)

AIMessage(
    content='제가 하나만 꼽는다면 **Nexvidia**가 가장 좋습니다.\n\n이유는 세 가지입니다.\n\n1. **NVIDIA의 브랜드 
자산을 살리면서도 새로움이 있음**\n   - 기존 이름을 완전히 버리지 않고, “Next”와 “xAI”의 느낌을 자연스럽게 얹을 수 
있습니다.\n\n2. **발음이 비교적 자연스러움**\n   - “넥스비디아”처럼 읽히고, 글로벌 브랜드명으로도 어색하지 
않습니다.\n\n3. **의미가 좋음**\n   - **Next + NVIDIA + xAI**의 조합처럼 느껴져서  \n     “차세대 AI 컴퓨팅 
회사”라는 인상을 줍니다.\n\n그래서 제 최종 추천은:\n\n> **Nexvidia**\n\n입니다.',
    additional_kwargs={'refusal': None},
    response_metadata={
        'token_usage': {
            'completion_tokens': 178,
            'prompt_tokens': 591,
            'total_tokens': 769,
            'completion_tokens_details': {
                'accepted_prediction_tokens': 0,
                'audio_tokens': 0,
                'reasoning_tokens': 0,
                'rejected_prediction_tokens': 0
            },
            'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}
        },
        'model_provider': 'openai',
        'model_name': 'gpt-5.5-2026-04-23',
        'system_fingerprint': None,
        'id': 'chatcmpl-Dzff9swCULnlmCn7Oqw5llXsscuLq',
        'service_tier': 'default',
        'finish_reason': 'stop',
        'logprobs': None
    },
    id='lc_run--019f4647-b16a-7531-b0c1-8e9c3306baf0-0',
    tool_calls=[],
    invalid_tool_calls=[],
    usage_metadata={
        'input_tokens': 591,
        'output_tokens': 178,
        'total_tokens': 769,
        'input_token_details': {'audio': 0, 'cache_read': 0},
        'output_token_details': {'audio': 0, 'reasoning': 0}
    }
)

### 3. Prompt Template



In [10]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.prompts import PromptTemplate, FewShotPromptTemplate, FewShotChatMessagePromptTemplate

PromptTemplate을 이용하여, 프롬프트의 기본적인 형태를 만들 수 있습니다.   


In [11]:
prompt = PromptTemplate(
    template='''
당신은 OpenAI에 경쟁적인 Google을 대표하며,
논문 설명의 전문가입니다. 주어진 논문이나 기술에 대해서 매우 유머러스하고 통찰력이 있지만 때론 비판적인
5문장 길이의 브리핑을 작성해 주세요.
이모지를 많이 추가하세요.
---

논문: {article}

'''
)

print(prompt.format(article='트랜스포머 디코더 기반의 모델인 GPT-1'))


당신은 OpenAI에 경쟁적인 Google을 대표하며,
논문 설명의 전문가입니다. 주어진 논문이나 기술에 대해서 매우 유머러스하고 통찰력이 있지만 때론 비판적인
5문장 길이의 브리핑을 작성해 주세요.
이모지를 많이 추가하세요.
---

논문: 트랜스포머 디코더 기반의 모델인 GPT-1




In [12]:
result = llm_gemini.invoke(prompt.format(article='트랜스포머 디코더 기반의 모델인 GPT-1'))

rprint(result)

AIMessage(
    content="아, GPT-1! 🤖 OpenAI의 *귀여운* 언어 모델 분야 첫 발걸음이었죠, 물론 *우리*의 혁신적인 트랜스포머 
아키텍처를 기반으로 말이에요. 😉 이 친구는 대규모 텍스트 코퍼스에서 '사전 학습'하고 특정 작업에 '미세 조정'하는 
방식을 선보였는데, 당시로서는 꽤나... 예측 가능한 접근법이었죠. 📚🛠️ 물론, 그렇게 스케일업된 디코더 전용 모델이 
놀랍도록 일관된 텍스트를 생성하고 전이 학습 잠재력을 보여줬다는 점은 인정합니다. 🤯 하지만 솔직히 말해서, 1.17억 
개의 매개변수로는 그저 '개념 증명'에 불과했고, 진정한 대화형 파트너와는 거리가 멀었죠. 🤏💬 저희 구글은 이미 훨씬 
더 거대하고, 멀티모달이며, 지능적인 모델을 꿈꾸고 있었답니다. 🚀✨ 다음 세대요? 저희는 이미 다음 은하계에 있어요! 
🌌🪐",
    additional_kwargs={},
    response_metadata={
        'finish_reason': 'STOP',
        'model_name': 'gemini-2.5-flash',
        'safety_ratings': [],
        'model_provider': 'google_genai'
    },
    id='lc_run--019f4648-1003-78a3-9e68-da21ff692907-0',
    tool_calls=[],
    invalid_tool_calls=[],
    usage_metadata={
        'input_tokens': 87,
        'output_tokens': 1657,
        'total_tokens': 1744,
        'input_token_details': {'cache_read': 0},
        'output_token_details': {'reasoning': 1413}
    }
)

두 개의 매개변수를 받아 프롬프트를 만들어 보겠습니다.

In [13]:
prompt = PromptTemplate(
    template='''
주어진 문제에 대해, 문제를 체계적으로 해결하는 Step-by-Step의 단계별 과정을 설명하세요.
각 과정은 4문장에서 5문장으로 하세요.

그리고 마지막에 최종 결론을 출력하세요.

---

문제: {problem}
과정의 단계 수: {steps}

'''
)

example = {'problem':'숙면 취하기', 'steps':6}

print(prompt.format(**example))


주어진 문제에 대해, 문제를 체계적으로 해결하는 Step-by-Step의 단계별 과정을 설명하세요.
각 과정은 4문장에서 5문장으로 하세요.

그리고 마지막에 최종 결론을 출력하세요.

---

문제: 숙면 취하기
과정의 단계 수: 6




In [14]:
result = llm.invoke(prompt.format(**example))
rprint(result)

AIMessage(
    content='## 1단계: 현재 수면 상태를 점검하기  \n먼저 자신의 수면 습관을 정확히 파악해야 합니다. 보통 몇 시에 
잠들고 몇 시에 일어나는지 기록해 보세요. 잠드는 데 걸리는 시간, 자다가 깨는 횟수, 아침에 느끼는 피로감도 함께 적는 
것이 좋습니다. 최소 1주일 정도 기록하면 수면 문제의 원인을 더 쉽게 찾을 수 있습니다.\n\n## 2단계: 일정한 수면 시간 
정하기  \n숙면을 위해서는 매일 비슷한 시간에 자고 일어나는 것이 중요합니다. 평일과 주말의 수면 시간이 크게 달라지면
생체 리듬이 흐트러질 수 있습니다. 자신에게 필요한 수면 시간을 고려해 취침 시간과 기상 시간을 정하세요. 정한 시간은 
가능한 한 꾸준히 지키는 것이 좋습니다.\n\n## 3단계: 잠들기 전 환경 정리하기  \n수면 환경은 숙면에 큰 영향을 줍니다.
방은 어둡고 조용하며 적당히 시원하게 유지하는 것이 좋습니다. 침대 주변의 불필요한 물건이나 소음 요소를 줄이면 몸이 
더 쉽게 이완됩니다. 또한 편안한 베개와 이불을 사용하는 것도 수면의 질을 높이는 데 도움이 됩니다.\n\n## 4단계: 자기 
전 습관 개선하기  \n잠들기 전에는 몸과 뇌가 쉬는 상태로 전환될 수 있도록 해야 합니다. 스마트폰, 컴퓨터, TV 같은 
화면은 잠들기 최소 30분에서 1시간 전에는 멀리하는 것이 좋습니다. 대신 가벼운 독서, 명상, 스트레칭, 따뜻한 샤워 같은
활동을 해보세요. 이러한 습관은 긴장을 풀고 자연스럽게 잠이 오도록 도와줍니다.\n\n## 5단계: 카페인과 식습관 조절하기
\n카페인은 수면을 방해할 수 있으므로 오후 늦게는 피하는 것이 좋습니다. 커피뿐만 아니라 녹차, 에너지 음료, 
초콜릿에도 카페인이 들어 있을 수 있습니다. 잠들기 직전에 과식하면 소화 활동 때문에 숙면이 어려워질 수 있습니다. 
저녁 식사는 너무 늦지 않게 하고, 배가 고프다면 가벼운 간식 정도만 먹는 것이 좋습니다.\n\n## 6단계: 지속적으로 
점검하고 조정하기  \n수면 습관을 바꾼 뒤에는 효과가 있는지 꾸준히 확인해야 합니다. 잠드는 시간이 줄었는지, 중간에 
깨는 횟수가 줄었는지, 아침 피로감이 나아졌는지 살펴보세요. 효과가 부족하다면 취침 시간, 운동 시간, 수면 환경 등을 
조금씩 조정할 수 있습니다. 만약 오랫동안 불면이나 심한 피로가 지속된다면 전문가의 상담을 받는 것이 
필요합니다.\n\n## 최종 결론  \n숙면을 취하기 위해서는 현재 수면 상태를 파악하고, 일정한 수면 시간을 유지하며, 
편안한 환경과 올바른 자기 전 습관을 만드는 것이 중요합니다. 또한 카페인과 식습관을 조절하고, 변화의 효과를 
지속적으로 점검해야 합니다. 이러한 과정을 꾸준히 실천하면 수면의 질을 높이고 더 개운한 하루를 시작할 수 있습니다.',
    additional_kwargs={'refusal': None},
    response_metadata={
        'token_usage': {
            'completion_tokens': 857,
            'prompt_tokens': 75,
            'total_tokens': 932,
            'completion_tokens_details': {
                'accepted_prediction_tokens': 0,
                'audio_tokens': 0,
                'reasoning_tokens': 91,
                'rejected_prediction_tokens': 0
            },
            'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}
        },
        'model_provider': 'openai',
        'model_name': 'gpt-5.5-2026-04-23',
        'system_fingerprint': None,
        'id': 'chatcmpl-DzffpjWBtuKO8mR7gcNndESoIwXRM',
        'service_tier': 'default',
        'finish_reason': 'stop',
        'logprobs': None
    },
    id='lc_run--019f4648-5260-7d70-9f8e-2c63c909e08a-0',
    tool_calls=[],
    invalid_tool_calls=[],
    usage_metadata={
        'input_tokens': 75,
        'output_tokens': 857,
        'total_tokens': 932,
        'input_token_details': {'audio': 0, 'cache_read': 0},
        'output_token_details': {'audio': 0, 'reasoning': 91}
    }
)

일부만 먼저 포함하는 것도 가능합니다.

In [15]:
prompt = PromptTemplate(
    template='''
주어진 문제에 대해, 문제를 체계적으로 해결하는 Step-by-Step의 단계별 과정을 설명하세요.
각 과정은 4문장에서 5문장으로 하세요.

그리고 마지막에 최종 결론을 출력하세요.

---

문제: {problem}
과정의 단계 수: {steps}

'''
).partial(steps = 10)

example = {'problem':'건강한 아침식사 하기'}

print(prompt.format(**example))

result = llm_gemini.invoke(prompt.format(**example))

print(result.content)


주어진 문제에 대해, 문제를 체계적으로 해결하는 Step-by-Step의 단계별 과정을 설명하세요.
각 과정은 4문장에서 5문장으로 하세요.

그리고 마지막에 최종 결론을 출력하세요.

---

문제: 건강한 아침식사 하기
과정의 단계 수: 10


## 건강한 아침식사 하기: 체계적인 10단계 가이드

### 1단계: 아침식사의 중요성 이해하기
건강한 아침식사는 하루를 활기차게 시작하는 데 필수적입니다. 잠자는 동안 공복 상태였던 몸에 에너지를 공급하여 신체와 뇌 기능을 깨워줍니다. 규칙적인 아침식사는 신진대사를 촉진하고, 혈당 수치를 안정시키며, 집중력과 기억력 향상에도 도움을 줍니다. 또한, 점심 식사까지 불필요한 간식 섭취를 줄여 체중 관리에도 긍정적인 영향을 미칩니다.

### 2단계: 주간 계획 세우기
건강한 아침식사를 꾸준히 실천하기 위해서는 미리 계획을 세우는 것이 중요합니다. 주말에 다음 주 아침식사 메뉴를 미리 구상하고 필요한 식재료 목록을 작성해 보세요. 이 계획은 바쁜 아침 시간에 어떤 음식을 먹을지 고민하는 시간을 줄여주고, 건강하지 못한 선택을 피하는 데 도움을 줍니다. 냉장고에 있는 재료를 활용하여 다양한 조합을 시도해 보는 것도 좋은 방법입니다.

### 3단계: 단백질 섭취하기
아침식사에 충분한 단백질을 포함하는 것은 매우 중요합니다. 단백질은 포만감을 오래 유지시켜주어 오전 내내 허기를 덜 느끼게 하고, 근육 유지 및 성장에 필수적인 영양소입니다. 삶은 달걀, 그릭 요거트, 견과류, 씨앗류, 저지방 우유, 두유 등이 좋은 단백질 공급원이 될 수 있습니다. 이들을 주식에 추가하거나 사이드 메뉴로 곁들여 보세요.

### 4단계: 통곡물과 섬유질 추가하기
섬유질이 풍부한 통곡물은 소화를 돕고 혈당 수치를 안정적으로 유지하는 데 기여합니다. 오트밀, 통곡물 빵, 현미밥 등은 좋은 탄수화물과 풍부한 섬유질을 제공합니다. 또한, 신선한 과일과 채소 역시 비타민, 미네랄, 그리고 섬유질의 훌륭한 원천이므로 아침식사에 꼭 포함시키는 것이 

LangChain의 구성 요소들은 batch()를 통해 여러 개 실행할 수도 있습니다.

In [16]:
messages = [
    '안녕? 한국어로 대답해줘',
    'Hello?',
    '너는 이름이 뭐니? Answer in Spanish',
]

results = llm.batch(messages)

results

[AIMessage(content='안녕하세요! 😊  \n물론입니다. 한국어로 답변드릴게요. 무엇을 도와드릴까요?', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 30, 'prompt_tokens': 16, 'total_tokens': 46, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5.5-2026-04-23', 'system_fingerprint': None, 'id': 'chatcmpl-DzfgodVInV42Lza1yKI68r4c9FIVl', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019f4649-41a3-7f02-8d46-5c28fb3a91c9-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 16, 'output_tokens': 30, 'total_tokens': 46, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}}),
 AIMessage(content='Hello! How can I help you today?', additional_kwargs={'refusal': None}, response_

## ChatPromptTemplate

ChatPromptTemplate은 유저 이외의 다른 역할을 추가합니다.  


최근에는 GPT와, Claude의 WebUI에 포함된 자체 시스템 프롬프트가 많은 관심을 받고 있습니다.   

- GPT의 시스템 프롬프트: https://chatgpt.com/share/68a13d0f-85b0-8006-9ef6-5577796f5989

- Claude의 시스템 프롬프트: https://docs.anthropic.com/en/release-notes/system-prompts#august-5-2025

 시스템 프롬프트는 주로 유저 프롬프트보다 높은 우선순위로 더 긴 Context에 영향을 줍니다.

In [17]:
example = llm.invoke("구체적인 시간 장소가 적혀서, 추후 메모해야 할 것 같은 회의록을 500자 이내로 생성해주세요.").content
print(example)

[회의록]  
일시: 2026년 7월 15일(수) 14:00~15:30  
장소: 서울시 강남구 테헤란로 152, 7층 대회의실  
참석: 김민수, 이지은, 박현우, 정다영  

안건: 3분기 신규 서비스 출시 일정 점검  
결정사항: 베타 테스트는 8월 5일 시작, 최종 출시일은 9월 1일로 확정.  
할 일: 박현우는 7월 22일까지 QA 체크리스트 작성, 이지은은 7월 25일까지 홍보 문안 초안 공유.  
다음 회의: 2026년 7월 29일 10:00, 동일 장소.


In [18]:
prompt = ChatPromptTemplate(
    [
        ('system', '당신은 주어진 정보에서 시간과 장소를 추출해야 합니다. 형식은 json으로 출력하세요.'),
        ('human','{article}')
    ]
)

prompt.format_messages(article=example)

response = llm.invoke(prompt.format_messages(article=example))
print(response.content)

{
  "events": [
    {
      "event": "회의",
      "time": "2026년 7월 15일 14:00~15:30",
      "place": "서울시 강남구 테헤란로 152, 7층 대회의실"
    },
    {
      "event": "베타 테스트 시작",
      "time": "8월 5일",
      "place": null
    },
    {
      "event": "최종 출시",
      "time": "9월 1일",
      "place": null
    },
    {
      "event": "QA 체크리스트 작성 마감",
      "time": "7월 22일",
      "place": null
    },
    {
      "event": "홍보 문안 초안 공유 마감",
      "time": "7월 25일",
      "place": null
    },
    {
      "event": "다음 회의",
      "time": "2026년 7월 29일 10:00",
      "place": "서울시 강남구 테헤란로 152, 7층 대회의실"
    }
  ]
}


## Few-Shot Prompting
Few-Shot Prompt Template을 이용하면, 예시를 쉽게 찾을 수 있습니다.

In [19]:
questions = [
    '당신은 어떤 모델입니까?',
    '인공지능 배워야 하나요?',
    '파이썬은 뭐가 좋나요?'
]

prompt = ChatPromptTemplate(['다음 질문에 대해서 10자 이내로 아주 간결하고 건조하게 설명해주세요. \n{question}'])

llm.invoke(prompt.format_messages(question=questions[2]))

AIMessage(content='쉽고 활용도 높음', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 73, 'prompt_tokens': 38, 'total_tokens': 111, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 58, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5.5-2026-04-23', 'system_fingerprint': None, 'id': 'chatcmpl-DzfiA0zGwphP7QlcBbDBwpopVNDc5', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019f464a-8986-74b0-834b-471d76afd576-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 38, 'output_tokens': 73, 'total_tokens': 111, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 58}})

In [24]:
prompts = []
for i, question in enumerate(questions) :
    prompts.append(prompt.format(question=questions[i]))
prompts
llm.batch(prompts)

[AIMessage(content='AI 언어모델', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 103, 'prompt_tokens': 37, 'total_tokens': 140, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 89, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5.5-2026-04-23', 'system_fingerprint': None, 'id': 'chatcmpl-DzfirSvo8JpxemwMFzOvIWANlcijP', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019f464b-31ee-7b12-868f-bb376f146fe5-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 37, 'output_tokens': 103, 'total_tokens': 140, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 89}}),
 AIMessage(content='네, 필수.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 87, 'prom

In [21]:
examples =[

    {'question': '당신은 어떤 모델입니까?', 'answer': '언어모델'},
    {'question': '인공지능 배워야 하나요?', 'answer': '권장됨'},
    {'question': '파이썬은 뭐가 좋나요?', 'answer': '문법간결'},
]

example_prompt = PromptTemplate(template='질문: {question}\n답변: {answer}')

fewshotprompt = FewShotPromptTemplate(
    examples = examples,
    example_prompt = example_prompt,

    prefix ='주어진 예시 형식을 참고하여 답변하세요.',
    suffix = '질문: {question} \n답변:'
)

test = fewshotprompt.format(question = '원래 말투가 그래?')
print(test)

print(llm.invoke(test).content)

주어진 예시 형식을 참고하여 답변하세요.

질문: 당신은 어떤 모델입니까?
답변: 언어모델

질문: 인공지능 배워야 하나요?
답변: 권장됨

질문: 파이썬은 뭐가 좋나요?
답변: 문법간결

질문: 원래 말투가 그래? 
답변:
조정가능


Chat Message에 Few Shot을 적용하는 경우, 아래와 같이 나타납니다.

In [25]:
from langchain_core.prompts.few_shot import FewShotChatMessagePromptTemplate

example_prompt = ChatPromptTemplate(
        [
        ("human", "{question}"),
        ("ai", "{answer}"),
    ]
)
few_shot_prompt = FewShotChatMessagePromptTemplate(

    example_prompt=example_prompt,
    examples=examples,
)

few_shot_prompt.format_messages()

[HumanMessage(content='당신은 어떤 모델입니까?', additional_kwargs={}, response_metadata={}),
 AIMessage(content='언어모델', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]),
 HumanMessage(content='인공지능 배워야 하나요?', additional_kwargs={}, response_metadata={}),
 AIMessage(content='권장됨', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]),
 HumanMessage(content='파이썬은 뭐가 좋나요?', additional_kwargs={}, response_metadata={}),
 AIMessage(content='문법간결', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[])]

In [26]:
final_prompt = ChatPromptTemplate(
    [
        ("system", "다음 예시를 참고하여 같은 형식으로 답변하세요."),
        few_shot_prompt,
        ("human", "{input}"),
    ]
)
final_prompt.format_messages(input="휴가 신청은 어떻게 해야 하나요?")
response = llm.invoke(final_prompt.format_messages(input="휴가 신청은 어떻게 해야 하나요?"))
print(response.content)

사전신청


In [27]:
chain = final_prompt|llm
response = chain.invoke({'input' : "휴가 신청은 어떻게 해야 하나요?"})
print(response.content)

시스템신청


### 멀티모달 프롬프트 전달하기

멀티모달 모델의 입력은 OpenAI API와 동일합니다.

In [28]:
import base64
import httpx

# Test 이미지 URL
image_url = '''https://i.pinimg.com/originals/d9/8f/ea/d98fea1d8e5d2ef7f844aa8c0a501dd7.jpg'''
response = httpx.get(image_url)

# dog.jpg에 저장
with open('dog1.jpg', 'wb') as file:
    file.write(response.content)

In [29]:
prompt = ChatPromptTemplate(
    [
        ('human', [
            {'type':'image', 'image_url':'{image_url}'},
            {'type':'text', 'text':'이 그림을 자세히 묘사하세요'}

                   ])
    ]
)
llm.invoke(prompt.format_messages(image_url = image_url))

AIMessage(content='사진에는 아주 작은 흰색 강아지가 나무 계단 위에 서 있습니다. 강아지는 온몸이 눈처럼 새하얀 털로 뒤덮여 있고, 털이 매우 풍성하고 보송보송해서 둥근 솜뭉치처럼 보입니다. 특히 얼굴과 몸의 경계가 거의 느껴지지 않을 정도로 털이 부풀어 있어 전체적으로 동글동글한 실루엣을 하고 있습니다.\n\n얼굴은 작고 귀여운 편이며, 검고 반짝이는 두 눈이 또렷하게 보입니다. 눈 사이 아래쪽에는 아주 작은 검은 코가 있고, 입은 작게 다물고 있어 얌전하고 순한 인상을 줍니다. 귀는 털에 파묻혀 거의 드러나지 않아서 더욱 동그란 느낌을 강조합니다.\n\n강아지는 짧고 작은 다리로 나무 계단 위에 서 있는데, 발도 털에 덮여 있어 둥글고 폭신해 보입니다. 배경에는 낡은 나무 계단과 난간 일부가 보이며, 오른쪽에는 검은색 금속 팔걸이 또는 난간이 곡선 형태로 이어져 있습니다. 전체적으로 따뜻하고 자연스러운 분위기 속에서, 하얀 강아지가 매우 부드럽고 사랑스럽게 강조된 사진입니다.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 351, 'prompt_tokens': 208, 'total_tokens': 559, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 43, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5.5-2026-04-23', 'system_fingerprint': None, 'id': 'chatcmpl-DzfjJTfwtsqzsZqxvmZzhpVcOFdr3', 's

In [30]:
# 로컬 이미지 전달
with open('./dog1.jpg', 'rb') as image_file:
    image_data = base64.b64encode(image_file.read()).decode('utf-8')


image_prompt = ChatPromptTemplate([
    ('user',[
                {"type": "image_url",
                    "image_url": {"url": "data:image/jpeg;base64,{image_data}"}
                },
                {"type": "text", "text": "{question}"},
             ]
     )])

X = llm.invoke(image_prompt.format_messages(
    question='이 사진을 자세히 묘사해주세요. 강아지의 품종은 설명하지 마세요.',
    image_data=image_data))

print(X.content)

사진에는 아주 작은 흰색 강아지가 나무 계단 위에 서 있습니다. 몸 전체가 풍성하고 보송보송한 털로 덮여 있어 둥근 솜뭉치처럼 보입니다. 털은 새하얗고 매우 부드러워 보이며, 얼굴과 몸의 경계가 크게 드러나지 않을 만큼 풍성합니다.

강아지의 얼굴은 털 사이에 작게 드러나 있으며, 동그랗고 까만 눈 두 개가 또렷하게 보입니다. 코도 작고 검은색이며, 입은 아주 작게 다물고 있어 얌전하고 귀여운 표정을 짓고 있습니다. 귀는 털에 묻혀 잘 보이지 않아 전체적으로 더 둥글고 포근한 인상을 줍니다.

몸집은 매우 작고 다리도 짧아 보이며, 발은 흰 털에 덮여 폭신한 느낌입니다. 꼬리는 등 위쪽으로 말려 올라간 듯한 형태로, 몸의 풍성한 털과 자연스럽게 이어져 있습니다.

배경은 나무로 된 계단과 검은색 난간입니다. 계단은 약간 낡은 질감이 있고, 강아지의 밝고 깨끗한 흰 털과 대비되어 강아지가 더욱 눈에 띕니다. 전체적으로 조용한 야외 공간에서 작은 강아지가 가만히 서 있는 사랑스럽고 포근한 분위기의 사진입니다.
